# Python-to-Select-Languages Porting Notebook

Port Python code to **C++, C, Rust, Java, and JavaScript** using an LLM and the week4 system knowledge base. The UI is built with Gradio.

- **System info** (`retrieve_system_info`) and **Rust toolchain info** (`rust_toolchain_info`) are passed into the prompt so the model can tailor code and compile/run commands for your machine.
- **Time comparison**: Run the original Python and each ported version, then see execution times and speedup vs Python.

In [14]:
# Setup: imports
import os
import io
import sys
import subprocess
import time
from dotenv import load_dotenv
from openai import OpenAI
import gradio as gr

In [15]:
load_dotenv(override=True)
openai_api_key = os.getenv("OPENAI_API_KEY")

openrouter_api_key = os.getenv("OPENROUTER_API_KEY")

if openai_api_key:
    print(f"OpenAI API Key exists (begins {openai_api_key[:8]}...)")
else:
    print("OpenAI API Key not set")
if openrouter_api_key:
    print(f"OpenRouter API Key exists (begins {openrouter_api_key[:6]}...)")
else:
    print("OpenRouter API Key not set")

OpenAI API Key exists (begins sk-proj-...)
OpenRouter API Key exists (begins sk-or-...)


In [ ]:
# Connect to client libraries

ollama_url = "http://localhost:11434/v1"
openrouter_url = "https://openrouter.ai/api/v1"


ollama = OpenAI(api_key="ollama", base_url=ollama_url)
openrouter = OpenAI(api_key=openrouter_api_key, base_url=openrouter_url) if openrouter_api_key else None

models = ["openai/gpt-5-nano", "openai/gpt-4.1-mini", "anthropic/claude-sonnet-4.5", "google/gemini-2.5-pro", "qwen/qwen2.5-coder", "deepseek/deepseek-coder-v2", "qwen/qwen3-coder-30b-a3b-instruct", "x-ai/grok-4-fast"]
clients = {
    "openai/gpt-5-nano": openrouter,
    "openai/gpt-4.1-mini": openrouter,
    "anthropic/claude-sonnet-4.5": openrouter,
    "google/gemini-2.0-flash": openrouter,
    "qwen/qwen2.5-coder": ollama,
    "deepseek/deepseek-coder-v2": ollama,
    "qwen/qwen3-coder-30b-a3b-instruct": openrouter,
    "x-ai/grok-4-fast": openrouter,
}
# Drop models whose client is None
models = [m for m in models if clients.get(m) is not None]
if not models:
    models = ["gpt-4o"]
    clients = {"gpt-4o": openrouter}

Run the following cells from the **week4** directory (or ensure `week4` is on `sys.path`) so that `system_info` can be imported.

In [26]:
# Load knowledge base (run from week4 or ensure week4 is on sys.path)
from system_info import retrieve_system_info, rust_toolchain_info

system_info = retrieve_system_info()
rust_info = rust_toolchain_info()
print("System info and Rust toolchain info loaded.")

System info and Rust toolchain info loaded.


In [27]:
# Per-language config: extension, filename, compile_command, run_command
TARGET_LANGUAGES = ["C++", "C", "Rust", "Java", "JavaScript"]

LANG_CONFIG = {
    "C++": {
        "ext": "cpp",
        "filename": "main.cpp",
        "compile": ["clang++", "-std=c++17", "-Ofast", "-mcpu=native", "main.cpp", "-o", "main"],
        "run": ["./main"],
    },
    "C": {
        "ext": "c",
        "filename": "main.c",
        "compile": ["clang", "-std=c11", "-O2", "main.c", "-o", "main"],
        "run": ["./main"],
    },
    "Rust": {
        "ext": "rs",
        "filename": "main.rs",
        "compile": ["rustc", "-C", "opt-level=3", "-C", "target-cpu=native", "main.rs", "-o", "main"],
        "run": ["./main"],
    },
    "Java": {
        "ext": "java",
        "filename": "Main.java",
        "compile": ["javac", "Main.java"],
        "run": ["java", "Main"],
    },
    "JavaScript": {
        "ext": "js",
        "filename": "main.js",
        "compile": [],
        "run": ["node", "main.js"],
    },
}

In [28]:
# Prompts: system and user with system_info (and rust_info for Rust)
def system_prompt_for(language):
    return f"""Your task is to convert Python code into high-performance {language} code.
Respond only with {language} code. Do not provide any explanation other than occasional comments.
The {language} response needs to produce identical output in the fastest possible time.
Note: Optimize for speed, everything else is secondary."""

def user_prompt_for(python_code, target_language):
    cfg = LANG_CONFIG[target_language]
    ext = cfg["ext"]
    compile_cmd = cfg["compile"]
    run_cmd = cfg["run"]
    base = f"""Port this Python code to {target_language} with identical output and best performance.
System information:
{system_info}
"""
    if target_language == "Rust":
        base += f"\nRust toolchain information:\n{rust_info}\n"
    base += f"""
The code will be written to {cfg['filename']} and then compiled/run as:
compile: {compile_cmd}
run: {run_cmd}
Respond only with {target_language} code.

Python code to port:

```python
{python_code}
```
"""
    return base

def messages_for(python_code, target_language):
    return [
        {"role": "system", "content": system_prompt_for(target_language)},
        {"role": "user", "content": user_prompt_for(python_code, target_language)},
    ]

In [29]:
# Core: clean_code, write_output, port
def clean_code(raw_reply, target_language):
    """Strip markdown code fences and language tags."""
    tags = ["cpp", "c++", "c", "rust", "java", "javascript", "js"]
    s = raw_reply.strip()
    for tag in tags:
        s = s.replace(f"```{tag}", "").replace(f"```{tag}\n", "")
    s = s.replace("```", "").strip()
    lines = s.splitlines()
    if lines and lines[0].strip().lower() in tags:
        lines = lines[1:]
    return "\n".join(lines)

def write_output(code, target_language):
    cfg = LANG_CONFIG[target_language]
    path = cfg["filename"]
    with open(path, "w") as f:
        f.write(code)
    return path

def port(model, python_code, target_language):
    client = clients.get(model)
    if client is None:
        return f"Error: no client for model {model}"
    kwargs = {"model": model, "messages": messages_for(python_code, target_language)}
    if "gpt" in model.lower():
        kwargs["reasoning_effort"] = "low"
    try:
        response = client.chat.completions.create(**kwargs)
        reply = response.choices[0].message.content or ""
    except Exception as e:
        return f"Error: {e}"
    cleaned = clean_code(reply, target_language)
    write_output(cleaned, target_language)
    return cleaned

In [30]:
# Run, time, and compare
def run_python(code):
    """Execute Python code in a restricted namespace. Returns (stdout_str, elapsed_seconds)."""
    buffer = io.StringIO()
    old_stdout = sys.stdout
    sys.stdout = buffer
    start = time.perf_counter()
    try:
        exec(code, {"__builtins__": __builtins__})
        output = buffer.getvalue()
    except Exception as e:
        output = f"Error: {e}"
    finally:
        sys.stdout = old_stdout
    elapsed = time.perf_counter() - start
    return (output, elapsed)

def compile_and_run(target_language):
    """Compile (if any) and run the ported code for target_language. Returns (stdout_str, elapsed_seconds) or (None, None) on failure."""
    cfg = LANG_CONFIG[target_language]
    compile_cmd = cfg["compile"]
    run_cmd = cfg["run"]
    if compile_cmd:
        try:
            subprocess.run(compile_cmd, check=True, text=True, capture_output=True, timeout=60)
        except subprocess.CalledProcessError as e:
            return (None, None)
        except Exception:
            return (None, None)
    start = time.perf_counter()
    try:
        result = subprocess.run(run_cmd, capture_output=True, text=True, timeout=120)
        stdout = result.stdout or result.stderr or ""
    except Exception as e:
        stdout = str(e)
    elapsed = time.perf_counter() - start
    return (stdout, elapsed)

def run_all_and_compare(python_code, ported_code_by_lang):
    """Run Python once and each ported language; return timing table as string."""
    out_py, time_py = run_python(python_code)
    rows = [("Python", time_py, 1.0)]
    for lang, code in ported_code_by_lang.items():
        if not code or not code.strip():
            rows.append((lang, None, None))
            continue
        write_output(clean_code(code, lang), lang)
        out_lang, time_lang = compile_and_run(lang)
        if time_lang is not None and time_py > 0:
            speedup = time_py / time_lang
            rows.append((lang, time_lang, speedup))
        else:
            rows.append((lang, None, None))
    lines = ["Language      | Time (s)  | Speedup vs Python", "-" * 45]
    for lang, t, speedup in rows:
        if t is not None and speedup is not None:
            lines.append(f"{lang:13} | {t:9.6f} | {speedup:.2f}x")
        else:
            lines.append(f"{lang:13} | (failed)   | -")
    return "\n".join(lines)

In [31]:
# Example Python code (pi calculation)
EXAMPLE_PYTHON = '''
import time

def calculate(iterations, param1, param2):
    result = 1.0
    for i in range(1, iterations+1):
        j = i * param1 - param2
        result -= (1/j)
        j = i * param1 + param2
        result += (1/j)
    return result

start_time = time.time()
result = calculate(200_000_000, 4, 1) * 4
end_time = time.time()

print(f"Result: {result:.12f}")
print(f"Execution Time: {(end_time - start_time):.6f} seconds")
'''

In [32]:
def run_and_compare_times(python_code, target_language):
    """Run Python and the ported target language; return timing and speedup."""
    if not python_code or not python_code.strip():
        return "Enter Python code and port it first."
    out_py, time_py = run_python(python_code)
    out_lang, time_lang = compile_and_run(target_language)
    if time_lang is None:
        return f"Python: {time_py:.6f}s\n{target_language}: (compile/run failed)"
    speedup = time_py / time_lang if time_lang > 0 else 0
    return f"Python: {time_py:.6f}s\n{target_language}: {time_lang:.6f}s ({speedup:.2f}x speedup)"

In [33]:
def compare_all_languages(python_code, model_name):
    """Port to all five languages then run and return timing table. (Uses 5 LLM calls.)"""
    if not python_code or not python_code.strip():
        return "Enter Python code first."
    ported = {}
    for lang in TARGET_LANGUAGES:
        try:
            code = port(model_name, python_code, lang)
            ported[lang] = code
        except Exception as e:
            ported[lang] = f"# Error: {e}"
    return run_all_and_compare(python_code, ported)

# Gradio UI: Port, Run & compare times, Compare all languages
with gr.Blocks(title="Python to Languages Porting") as ui:
    with gr.Row():
        python_in = gr.Textbox(label="Python code", lines=22, value=EXAMPLE_PYTHON)
        ported_out = gr.Textbox(label="Ported code", lines=22)
    with gr.Row():
        model_dd = gr.Dropdown(choices=models, label="Model", value=models[0])
        lang_dd = gr.Dropdown(choices=TARGET_LANGUAGES, label="Target language", value="C++")
        port_btn = gr.Button("Port")
    with gr.Row():
        compare_btn = gr.Button("Run & compare times")
        compare_out = gr.Textbox(label="Timing (Python vs ported)", lines=5)
    with gr.Row():
        compare_all_btn = gr.Button("Compare all languages (ports to all 5, then runs)")
        compare_all_out = gr.Textbox(label="Timing table (all languages)", lines=10)
    port_btn.click(
        fn=lambda model, code, lang: port(model, code, lang),
        inputs=[model_dd, python_in, lang_dd],
        outputs=[ported_out],
    )
    compare_btn.click(
        fn=run_and_compare_times,
        inputs=[python_in, lang_dd],
        outputs=[compare_out],
    )
    compare_all_btn.click(
        fn=compare_all_languages,
        inputs=[python_in, model_dd],
        outputs=[compare_all_out],
    )

ui.launch(inbrowser=True)

* Running on local URL:  http://127.0.0.1:7863
* To create a public link, set `share=True` in `launch()`.
